# Tree/Linear Models Project — Regression, Regularization, SVM & SHAP

**My notes for this notebook:**
This notebook walks through the full workflow for the `linear_models` project:
data generation → EDA → Linear Regression → Ridge → Lasso → SHAP explainability →
Logistic Regression (classification) → SVM with different kernels.

The point of this project isn't just "call `.fit()` and get a number" — it's to actually
*see* what regularization (Ridge/Lasso) does to a model, and *see* how a linear decision
boundary compares to a kernel-based one (SVM). So every section has a plot, and a note on
what the plot is telling us.


## 0. Setup

Installing/importing everything we'll need across all tasks. `shap` usually isn't
pre-installed on Colab, so I install it first.


In [ ]:
!pip install shap -q


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from sklearn import linear_model, svm, metrics
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification, make_gaussian_quantiles
from sklearn.metrics import accuracy_score, classification_report

import shap

np.random.seed(42)


## 1. Dataset #1 — Multicollinearity dataset (Tasks 0, 1, 2, 4a)

**Why this dataset is built this way:**
- `X1` and `X2` are almost the same variable (`X2 = X1 + tiny noise`) — this is
  **multicollinearity** on purpose. In real data this happens all the time
  (e.g. "height in cm" and "height in inches").
- `X3` is independent of `X1`/`X2`.
- `y` only actually depends on `X1` and `X3` (`y = 4*X1 + 3*X3 + noise`) — `X2` is a
  "fake" feature that just rides along with `X1`.

**Why it matters:** with two nearly-identical features, plain Linear Regression can't tell
which one "deserves" the credit for predicting `y`, so it splits/exaggerates the
coefficients between `X1` and `X2` in an unstable way. That's exactly the problem Ridge
Regression (L2 regularization) is designed to fix — it will show up clearly once we
compare the two.


In [ ]:
np.random.seed(42)

X1 = np.random.rand(200) * 10
X2 = X1 + np.random.normal(0, 0.05, 200)
X3 = np.random.rand(200) * 5
X_mc = np.column_stack([X1, X2, X3])
y_mc = 4*X1 + 3*X3 + np.random.normal(0, 5, 200)

print("X_mc shape:", X_mc.shape)
print("y_mc shape:", y_mc.shape)


### 1.1 EDA — visualizing dataset #1

**Why EDA before modeling:** before I trust any model output, I want to *see* the
relationships with my own eyes. Two things I'm checking here:

1. **3D scatter of X1, X2, X3** — this should visually confirm that X1 and X2 sit almost
   on a perfect diagonal line (i.e., they're basically the same feature), while X3 is
   scattered independently.
2. **Each feature vs. y** — this should show X1 and X3 having a clear upward linear trend
   with y, while X2 (since it mirrors X1) will *also* look correlated with y — even though
   it's not actually a real driver of it. This is the visual trap that causes the
   multicollinearity problem.


In [ ]:
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

ax.scatter(X1, X2, X3, c='blue', alpha=0.7, edgecolor='k')
ax.set_xlabel('X1')
ax.set_ylabel('X2')
ax.set_zlabel('X3')
ax.set_title('3D Scatter Plot of Features X1, X2, and X3')
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(X_mc[:, 0], y_mc, color='blue', alpha=0.6, edgecolors='k')
axes[0].set_xlabel('X1'); axes[0].set_ylabel('y'); axes[0].set_title('X1 vs y')

axes[1].scatter(X_mc[:, 1], y_mc, color='green', alpha=0.6, edgecolors='k')
axes[1].set_xlabel('X2'); axes[1].set_ylabel('y'); axes[1].set_title('X2 vs y')

axes[2].scatter(X_mc[:, 2], y_mc, color='orange', alpha=0.6, edgecolors='k')
axes[2].set_xlabel('X3'); axes[2].set_ylabel('y'); axes[2].set_title('X3 vs y')

fig.suptitle('Exploring the Relationship Between Input Features (X1, X2, X3) and y', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()


### 1.2 Train/test split

Standard 70/30 split, fixed `random_state` so results are reproducible every time I rerun
the notebook.


In [ ]:
X_train_mc, X_test_mc, y_train_mc, y_test_mc = train_test_split(
    X_mc, y_mc, test_size=0.3, random_state=42)

print(X_train_mc.shape, X_test_mc.shape)


## 2. Task 0 — Linear Regression

**What it does:** Ordinary Least Squares — fits a straight-line (hyperplane) relationship
between the features and `y` by minimizing squared error. No regularization at all, so
it will happily give large/unstable coefficients if features are correlated (like X1/X2
here).

**Why I write it as a function:** so it can just be imported and reused across every
main script/task, instead of retyping `linear_model.LinearRegression()` everywhere.


In [ ]:
def Linear_Regression():
    """
    Creates a Linear Regression model using scikit-learn.

    Returns:
        model: an untrained LinearRegression instance.
    """
    model = linear_model.LinearRegression()
    return model


In [ ]:
lr = Linear_Regression()
lr.fit(X_train_mc, y_train_mc)

print(lr.get_params())
print("\nLinear Regression Coefficients (for [X1, X2, X3]):", lr.coef_)
print("Linear Regression Intercept (bias term):", lr.intercept_)


**Note to self:** watch the coefficients for X1 and X2 here — since they're almost the
same variable, I expect the model to split credit between them unevenly (maybe even push
one coefficient in a weird/opposite direction), instead of cleanly assigning the true
weight (~4) to X1 and ~0 to X2. That instability is the whole reason Ridge exists.


## 3. Task 1 — Regression Evaluation Metrics

**Why I need this:** a single number (like just R²) doesn't tell the whole story. I want:

- **MSE** (Mean Squared Error) — penalizes big errors more (squared), sensitive to outliers.
- **RMSE** — same units as `y`, so it's actually interpretable ("on average, off by X").
- **MAE** (Mean Absolute Error) — more robust to outliers than MSE/RMSE.
- **R²** — how much of the variance in `y` is explained by the model (1.0 = perfect).

Bundling all four into one function means every model I evaluate later (Linear, Ridge,
Lasso, Logistic...) gets compared the same consistent way.


In [ ]:
def evaluation_metrics_for_regression(y_true, y_pred):
    """
    Computes common regression evaluation metrics.

    Args:
        y_true: array-like of true target values.
        y_pred: array-like of predicted target values.

    Returns:
        mse, rmse, mae, r2
    """
    mse = metrics.mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = metrics.mean_absolute_error(y_true, y_pred)
    r2 = metrics.r2_score(y_true, y_pred)

    return mse, rmse, mae, r2


In [ ]:
y_pred_lr = lr.predict(X_test_mc)
mse, rmse, mae, r2 = evaluation_metrics_for_regression(y_test_mc, y_pred_lr)

print("Linear Regression - Model Evaluation on Test Set\n")
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"R² Score: {r2:.4f}")


## 4. Task 2 — Ridge Regression (L2 Regularization)

**What it does differently from plain Linear Regression:** it adds a penalty term
proportional to the *sum of squared coefficients* to the loss function. This discourages
any single coefficient from getting too large, which directly attacks the instability
caused by multicollinearity (X1 vs X2). It **shrinks** coefficients toward zero, but
rarely makes them *exactly* zero — it spreads the "blame" more evenly instead of
eliminating features.

**Why `random_state` even matters for Ridge:** most Ridge solvers are deterministic, but
some (`'sag'`, `'saga'`) use randomness internally, so passing `random_state` keeps things
reproducible regardless of which solver ends up being used.


In [ ]:
def ridge_regression(random_state):
    """
    Creates a Ridge Regression model using scikit-learn.

    Ridge Regression extends ordinary linear regression by adding
    L2 regularization, which helps stabilize the model by shrinking
    large coefficients.

    Args:
        random_state: an integer used to set the random seed for
            reproducibility.

    Returns:
        model: an untrained Ridge regression model instance.
    """
    model = linear_model.Ridge(random_state=random_state)
    return model


In [ ]:
ridge = ridge_regression(random_state=42)
ridge.fit(X_train_mc, y_train_mc)

y_pred_ridge = ridge.predict(X_test_mc)

mse_lr, rmse_lr, mae_lr, r2_lr = evaluation_metrics_for_regression(y_test_mc, y_pred_lr)
mse_ridge, rmse_ridge, mae_ridge, r2_ridge = evaluation_metrics_for_regression(y_test_mc, y_pred_ridge)

print("=== Model Performance with Multicollinearity ===")
print(f"Linear Regression R²: {r2_lr:.4f}")
print(f"Ridge Regression R²: {r2_ridge:.4f}")

print("\nLinear Regression coefficients [X1, X2, X3]:", lr.coef_)
print("Ridge Regression coefficients  [X1, X2, X3]:", ridge.coef_)


**What to look for in the coefficients printed above:** Ridge's [X1, X2, X3]
coefficients should look more "sane"/stable than Linear Regression's — less extreme
swings between X1 and X2 — even if overall R² doesn't change dramatically. Regularization
is mainly about *stability/generalization*, not always about a huge accuracy jump on one
test set.


In [ ]:
plt.figure(figsize=(9, 6))
plt.plot([y_test_mc.min(), y_test_mc.max()], [y_test_mc.min(), y_test_mc.max()],
         'k--', label='Perfect Prediction')

plt.scatter(y_test_mc, y_pred_lr, alpha=1, c='red',
            label=f'Linear Regression (R²={r2_lr:.4f})')
plt.scatter(y_test_mc, y_pred_ridge, alpha=1, c='green',
            label=f'Ridge Regression (R²={r2_ridge:.4f})')

plt.title('True vs Predicted Values (Ridge vs Linear Regression)', fontsize=14)
plt.xlabel('True Values', fontsize=12)
plt.ylabel('Predicted Values', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 5. Dataset #2 — High-dimensional, sparse-signal dataset (Tasks 3, 4b)

**Why a new dataset here:** the multicollinearity dataset was about *correlated*
features. Lasso's superpower is different — it's about *irrelevant* features. So this
second dataset is built to showcase that:

- 100 samples, but **50 features**, all random noise (`X = np.random.randn(100, 50)`).
- Only the **first 5** features actually influence `y` (coefficients `[5, 4, 3, 2, 1]`);
  the other 45 are pure noise with zero true effect.

**Why it matters:** this is a classic "more features than truly matter" situation —
common in real-world data (e.g. hundreds of survey questions, only a handful actually
predictive). Plain Linear Regression has no way to *ignore* irrelevant features, so it
will overfit to noise. Lasso's L1 penalty can force irrelevant coefficients to exactly
zero, effectively doing automatic feature selection.


In [ ]:
np.random.seed(42)

X_sparse = np.random.randn(100, 50)
true_coef = np.zeros(50)
true_coef[:5] = [5, 4, 3, 2, 1]
y_sparse = X_sparse @ true_coef + np.random.normal(0, 3, 100)

X_train_sp, X_test_sp, y_train_sp, y_test_sp = train_test_split(
    X_sparse, y_sparse, test_size=0.3, random_state=42)

print(X_train_sp.shape, X_test_sp.shape)


## 6. Task 3 — Lasso Regression (L1 Regularization)

**What it does differently from Ridge:** Ridge (L2) shrinks coefficients but keeps them
all nonzero. Lasso (L1) can push coefficients to **exactly zero** — meaning it literally
drops features it decides aren't useful. That's why Lasso is often used for automatic
feature selection, not just stabilization.


In [ ]:
def lasso_regression(random_state):
    """
    Creates a Lasso Regression model using scikit-learn.

    Lasso Regression extends ordinary linear regression by adding
    L1 regularization, which helps simplify the model by forcing
    some coefficients to zero, enabling automatic feature selection.

    Args:
        random_state: an integer used to set the random seed for
            reproducibility.

    Returns:
        model: an untrained Lasso regression model instance.
    """
    model = linear_model.Lasso(random_state=random_state)
    return model


In [ ]:
lr_sp = Linear_Regression()
lr_sp.fit(X_train_sp, y_train_sp)

lasso = lasso_regression(random_state=42)
lasso.fit(X_train_sp, y_train_sp)

y_pred_lr_sp = lr_sp.predict(X_test_sp)
y_pred_lasso = lasso.predict(X_test_sp)

mse_lr_sp, rmse_lr_sp, mae_lr_sp, r2_lr_sp = evaluation_metrics_for_regression(y_test_sp, y_pred_lr_sp)
mse_lasso, rmse_lasso, mae_lasso, r2_lasso = evaluation_metrics_for_regression(y_test_sp, y_pred_lasso)

print(f"Linear Regression R²: {r2_lr_sp:.4f}")
print(f"Lasso Regression R²: {r2_lasso:.4f}")
print("\nTrue relevant features: 5 (defined during data generation)")
print(f"Linear uses {np.sum(lr_sp.coef_ != 0)}/50 features")
print(f"Lasso uses {np.sum(lasso.coef_ != 0)}/50 features")


**This is the key comparison of the whole section:** Linear Regression will use all
50/50 features (it has no mechanism to drop any), and will overfit to noise, hurting its
R² on the test set. Lasso should collapse down to close to the true 5 relevant features,
and get a *noticeably better* R² — proof that "fewer, correct features" beats "more,
noisy features."


In [ ]:
plt.figure(figsize=(9, 6))
plt.plot([y_test_sp.min(), y_test_sp.max()], [y_test_sp.min(), y_test_sp.max()],
         'k--', label='Perfect Prediction')

plt.scatter(y_test_sp, y_pred_lr_sp, alpha=1, c='red',
            label=f'Linear Regression (R²={r2_lr_sp:.3f})')
plt.scatter(y_test_sp, y_pred_lasso, alpha=1, c='green',
            label=f'Lasso Regression (R²={r2_lasso:.3f})')

plt.title('True vs Predicted Values (Lasso vs Linear Regression)', fontsize=14)
plt.xlabel('True Values', fontsize=12)
plt.ylabel('Predicted Values', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 7. Task 4 — SHAP: Model Explainability

**Why SHAP:** knowing the coefficients tells me the model's *global* rule, but SHAP tells
me, **for each individual prediction**, how much each feature pushed the prediction up or
down from the average. It's the difference between "X1 has coefficient 4" (global) and
"for this specific sample, X1 contributed +7.2 to the prediction" (local, additive,
comparable across features).

**How the explainer works here:** `shap.Explainer(model, X_train)` auto-detects that
these are linear models and uses an exact/efficient linear explainer under the hood —
no sampling approximation needed, unlike with tree-based or black-box models.


In [ ]:
def get_shap_explainer_and_values(model, X_train, X_test):
    """
    Creates a SHAP explainer and computes SHAP values.

    Args:
        model: a trained regression model.
        X_train: input data used to initialize the explainer
            (background dataset).
        X_test: input data to explain.

    Returns:
        explainer: SHAP explainer object.
        shap_values: SHAP values for the predictions on X_test.
    """
    explainer = shap.Explainer(model, X_train)
    shap_values = explainer(X_test)

    return explainer, shap_values


### 7.1 SHAP on the multicollinearity dataset — Linear vs Ridge

**What I expect:** Linear Regression's SHAP values for X1 and X2 should look erratic/
unbalanced (matching its unstable coefficients from Task 0/2), while Ridge's should be
more evenly and sensibly distributed between X1 and X2, and X3 should clearly matter in
both since it's a genuinely independent, relevant feature.


In [ ]:
feature_names_mc = ["X1", "X2", "X3"]

explainer_lr, shap_values_lr = get_shap_explainer_and_values(lr, X_train_mc, X_test_mc)
explainer_ridge, shap_values_ridge = get_shap_explainer_and_values(ridge, X_train_mc, X_test_mc)

shap_values_lr.feature_names = feature_names_mc
shap_values_ridge.feature_names = feature_names_mc

plt.figure(figsize=(15, 12))

plt.subplot(2, 2, 1)
shap.plots.bar(shap_values_lr, show=False)
plt.title("SHAP Feature Importance – Linear Regression")
plt.gca().set_xlabel("Mean Absolute SHAP Values Across Samples")

plt.subplot(2, 2, 2)
shap.plots.bar(shap_values_ridge, show=False)
plt.title("SHAP Feature Importance – Ridge Regression")
plt.gca().set_xlabel("Mean Absolute SHAP Values Across Samples")

plt.subplot(2, 2, 3)
shap.plots.beeswarm(shap_values_lr, show=False)
plt.title("SHAP Beeswarm Plot – Linear Regression")

plt.subplot(2, 2, 4)
shap.plots.beeswarm(shap_values_ridge, show=False)
plt.title("SHAP Beeswarm Plot – Ridge Regression")

plt.tight_layout()
plt.show()


### 7.2 SHAP on the sparse dataset — Linear vs Lasso

**What I expect:** Lasso's SHAP bar/beeswarm plots should show a handful of features
(X1–X5, the true signal) with real importance, and the rest hovering near zero. Linear
Regression's plot should be noisy and spread thin across many/all 50 features — visual
proof of the overfitting we already saw numerically in Task 3.


In [ ]:
feature_names_sp = [f"X{i}" for i in range(1, 51)]

explainer_lr_sp, shap_values_lr_sp = get_shap_explainer_and_values(lr_sp, X_train_sp, X_test_sp)
explainer_lasso, shap_values_lasso = get_shap_explainer_and_values(lasso, X_train_sp, X_test_sp)

shap_values_lr_sp.feature_names = feature_names_sp
shap_values_lasso.feature_names = feature_names_sp

plt.figure(figsize=(30, 18))

plt.subplot(2, 2, 1)
shap.plots.bar(shap_values_lr_sp, show=False)
plt.title("SHAP Feature Importance – Linear Regression")
plt.gca().set_xlabel("Mean Absolute SHAP Values Across Samples")

plt.subplot(2, 2, 2)
shap.plots.bar(shap_values_lasso, show=False)
plt.title("SHAP Feature Importance – Lasso Regression")
plt.gca().set_xlabel("Mean Absolute SHAP Values Across Samples")

plt.subplot(2, 2, 3)
shap.plots.beeswarm(shap_values_lr_sp, show=False)
plt.title("SHAP Beeswarm Plot – Linear Regression")

plt.subplot(2, 2, 4)
shap.plots.beeswarm(shap_values_lasso, show=False)
plt.title("SHAP Beeswarm Plot – Lasso Regression")

plt.tight_layout()
plt.subplots_adjust(left=0.2, right=0.8)
plt.show()


## 8. Task 5 — Logistic Regression (Classification)

**Switching gears:** everything above was regression (predicting a number). Now we move
to **classification** (predicting a category/class). Logistic Regression fits a logistic
(sigmoid) function to model the *probability* of a sample belonging to class 1, then
thresholds that probability (default 0.5) to make a hard 0/1 prediction.

**Dataset:** `make_classification` generates two clusters (one per class) in 2D space
that are reasonably separable — a good sanity-check case for a linear classifier.


In [ ]:
def Logistic_Regression_Model(random_state):
    """
    Creates a Logistic Regression model using scikit-learn.

    Logistic Regression performs binary classification by fitting
    a logistic function to the data.

    Args:
        random_state: an integer used to set the random seed for
            reproducibility.

    Returns:
        model: an untrained LogisticRegression instance.
    """
    model = linear_model.LogisticRegression(random_state=random_state)
    return model


In [ ]:
seed = 42
np.random.seed(seed)

X_clf, y_clf = make_classification(
    n_samples=1000,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_clusters_per_class=1,
    n_classes=2,
    random_state=seed
)

X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.3, random_state=seed)

log_model = Logistic_Regression_Model(random_state=seed)
log_model.fit(X_train_clf, y_train_clf)

y_pred_clf = log_model.predict(X_test_clf)

accuracy = accuracy_score(y_test_clf, y_pred_clf)
print("################ Logistic Regression Model ################\n")
print(f"Accuracy: {accuracy:.2f}\n")
print("Classification Report:\n")
print(classification_report(y_test_clf, y_pred_clf))


**Reading the classification report:** precision/recall/f1 per class tell me *how*
the model is wrong, not just *how often*. A high accuracy with lopsided precision/recall
between classes would hint at bias toward one class — worth watching for, especially on
imbalanced data (this dataset is balanced, so I mainly expect fairly even precision/recall
here).


In [ ]:
h = 0.02
x_min, x_max = X_clf[:, 0].min() - 0.5, X_clf[:, 0].max() + 0.5
y_min, y_max = X_clf[:, 1].min() - 0.5, X_clf[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
Z = log_model.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.figure(figsize=(18, 5))
plt.contourf(xx, yy, Z, alpha=0.4, cmap='viridis')
plt.scatter(X_clf[:, 0], X_clf[:, 1], c=y_clf, edgecolors='k', cmap='viridis', alpha=0.7)
plt.title("Decision Boundary – Understanding Class Separation in Feature Space", fontsize=14)
plt.xlabel("Feature 1", fontsize=12)
plt.ylabel("Feature 2", fontsize=12)
class0_patch = mpatches.Patch(color=plt.cm.viridis(0.0), label='Class 0')
class1_patch = mpatches.Patch(color=plt.cm.viridis(1.0), label='Class 1')
plt.legend(handles=[class0_patch, class1_patch], title="Classes")
plt.show()


**Why this plot matters:** the decision boundary is a straight line — that's the
fundamental nature of Logistic Regression, it can only separate classes linearly. This
will matter a lot in the next section when we hit a dataset that ISN'T linearly
separable.


In [ ]:
x_min, x_max = X_clf[:, 0].min() - 1, X_clf[:, 0].max() + 1
y_min, y_max = X_clf[:, 1].min() - 1, X_clf[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))

grid = np.c_[xx.ravel(), yy.ravel()]
prob = log_model.predict_proba(grid)[:, 1]
prob = prob.reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, prob, levels=50, cmap='viridis', alpha=0.6)
plt.colorbar(label='Predicted Probability')
plt.scatter(X_clf[y_clf == 0, 0], X_clf[y_clf == 0, 1], color='purple', edgecolor='k', label='Class 0')
plt.scatter(X_clf[y_clf == 1, 0], X_clf[y_clf == 1, 1], color='orange', edgecolor='k', label='Class 1')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Logistic Regression Predicted Probability Contour')
plt.legend()
plt.show()


**Note:** this probability contour is the *actual* thing Logistic Regression computes
internally — the hard 0/1 decision boundary from before is just this probability surface
sliced at the 0.5 threshold. Seeing the smooth gradient here makes that connection
concrete.


## 9. Task 6 — SVM Classifier with Different Kernels

**The point of this section:** show what happens when the classes are **not** linearly
separable, and how SVM kernels let us handle that.

**Dataset:** `make_gaussian_quantiles` creates 3 classes as *concentric* regions (like
rings around a center) — no straight line can separate them well. This is a deliberately
"hard" case for linear models.

**Kernels:**
- `'linear'` — same limitation as Logistic Regression, straight-line boundaries only.
- `'poly'` — can bend the boundary using polynomial combinations of features, more
  flexible than linear but not fully flexible.
- `'rbf'` (radial basis function) — measures similarity based on distance from points,
  which naturally creates *curved/circular* boundaries — perfect for this ring-shaped
  data.

**My expectation before running:** RBF should crush this dataset (high accuracy), poly
should do okay/better than linear, and both linear SVM and Logistic Regression should
struggle since neither can bend around the rings.


In [ ]:
def get_SVM_model(name, random_state):
    """
    Creates a Support Vector Machine (SVM) classifier with the
    specified kernel using scikit-learn.

    Args:
        name: a string indicating the type of kernel to use.
            Accepted values are:
                'linear': SVM model with a linear kernel.
                'poly': SVM model with a polynomial kernel.
                'rbf': SVM model with a radial basis function
                    (RBF) kernel.
        random_state: the seed used by the random number generator
            for reproducibility.

    Returns:
        model: an untrained SVC instance with the specified kernel.
    """
    model = svm.SVC(kernel=name, random_state=random_state)
    return model


In [ ]:
seed = 42
np.random.seed(seed)

X_svm, y_svm = make_gaussian_quantiles(
    n_samples=300,
    n_features=2,
    n_classes=3,
    random_state=seed)

X_train_svm, X_test_svm, y_train_svm, y_test_svm = train_test_split(
    X_svm, y_svm, stratify=y_svm, random_state=seed)

lr_svm_ds = Logistic_Regression_Model(seed).fit(X_train_svm, y_train_svm)
svm_linear = get_SVM_model("linear", seed).fit(X_train_svm, y_train_svm)
svm_poly = get_SVM_model("poly", seed).fit(X_train_svm, y_train_svm)
svm_rbf = get_SVM_model("rbf", seed).fit(X_train_svm, y_train_svm)

print("Logistic Regression classification report:\n",
      classification_report(y_test_svm, lr_svm_ds.predict(X_test_svm)))
print("SVM Linear classification report:\n",
      classification_report(y_test_svm, svm_linear.predict(X_test_svm)))
print("SVM Poly classification report:\n",
      classification_report(y_test_svm, svm_poly.predict(X_test_svm)))
print("SVM RBF classification report:\n",
      classification_report(y_test_svm, svm_rbf.predict(X_test_svm)))


In [ ]:
def plot_decision_boundary(model, title, subplot):
    h = 0.02
    x_min, x_max = X_svm[:, 0].min() - 1, X_svm[:, 0].max() + 1
    y_min, y_max = X_svm[:, 1].min() - 1, X_svm[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    subplot.contourf(xx, yy, Z, alpha=0.4, cmap='viridis')
    subplot.scatter(X_svm[:, 0], X_svm[:, 1], c=y_svm, cmap='viridis', edgecolor='k', s=40)
    subplot.set_title(title)
    subplot.set_xlabel("Feature 1")
    subplot.set_ylabel("Feature 2")

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
plot_decision_boundary(lr_svm_ds, "Logistic Regression", axes[0, 0])
plot_decision_boundary(svm_linear, "SVM (Linear Kernel)", axes[0, 1])
plot_decision_boundary(svm_poly, "SVM (Polynomial Kernel)", axes[1, 0])
plot_decision_boundary(svm_rbf, "SVM (RBF Kernel)", axes[1, 1])
plt.tight_layout()
plt.show()


**Reading the 4 panels together:** Logistic Regression and linear-kernel SVM should
draw straight-line-ish boundaries that cut right through the concentric rings, misclassifying
a lot of points. Poly should curve a bit more, catching more of the structure. RBF should
visibly wrap curved boundaries around each ring, matching the data's actual shape — and
its accuracy in the classification report above should be dramatically higher than the
other three.


## 10. Wrap-up notes

**Big picture takeaways from this notebook, in my own words:**

- **Linear Regression** is the baseline — no protection against multicollinearity or
  irrelevant features, so it's the "sees everything, filters nothing" model.
- **Ridge (L2)** fixes multicollinearity by *shrinking* correlated coefficients toward
  each other/zero, without eliminating them — good when I believe most features are at
  least somewhat useful.
- **Lasso (L1)** fixes "too many irrelevant features" by *zeroing out* coefficients
  entirely — good when I suspect only a subset of features actually matter (built-in
  feature selection).
- **SHAP** turns any of these linear models' global coefficients into per-sample,
  per-feature explanations — useful for explaining *individual* predictions, not just the
  model as a whole.
- **Logistic Regression** is the classification analogue of Linear Regression — same
  "straight line only" limitation, just applied to decision boundaries instead of
  continuous predictions.
- **SVM with RBF kernel** shows how switching from a linear decision boundary to a
  similarity/distance-based one lets a model capture much more complex (non-linear)
  class structure — at the cost of being less directly interpretable than the linear
  models above.

**Next steps I might explore later:** tuning `alpha` for Ridge/Lasso (regularization
strength), trying `GridSearchCV` for the SVM's `C`/`gamma` parameters, and comparing SHAP
explanations for the RBF SVM using `KernelExplainer` (since it's not a linear model, SHAP
has to approximate rather than compute exactly).
